# Ouro Training Continuation
**Steps:** 100 | **Checkpoint:** `https://hf.co/ouro-checkpoints/checkpoint.csf`

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save

Then: Runtime → Run all (Ctrl+F9)

In [ ]:
!pip install -q huggingface_hub zstandard
import subprocess, sys, json
import csf
from huggingface_hub import hf_hub_download, upload_file


In [ ]:
HF_REPO = "ouro-checkpoints"
CHECKPOINT_FILE = "checkpoint.csf"
STEPS = 100

local_csf = hf_hub_download(repo_id=HF_REPO, filename=CHECKPOINT_FILE, repo_type='model')
csf.unpack(local_csf, '/content/checkpoint')
print('Checkpoint unpacked.')


In [ ]:
result = subprocess.run([
    sys.executable, 'scripts/train_ouro.py',
    '--resume_from', '/content/checkpoint',
    f'--max_steps', str(STEPS),
    '--seq_len', '1536',
    '--output_dir', '/content/output',
], check=True)
print('Training complete.')


In [ ]:
manifest = csf.pack(['/content/output'], '/content/output.csf')
upload_file(
    path_or_fileobj='/content/output.csf',
    path_in_repo='output.csf',
    repo_id=HF_REPO,
    repo_type='model',
)
print(json.dumps({'status': 'done', 'steps': STEPS, 'sha256': manifest.get('footer_sha256')}))
